In [1]:
from openevolve import run_evolution
from openevolve.config import Config, LLMModelConfig
import time, importlib.util, asyncio
import nest_asyncio
import sys, io, os

try:
    asyncio.get_running_loop()
    asyncio.set_event_loop(asyncio.new_event_loop())
except RuntimeError:
    pass

os.environ["PYTHONIOENCODING"] = "utf-8"

# Safe fix (works in VSCode / Jupyter / normal Python)
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    sys.stderr.reconfigure(encoding='utf-8', errors='replace')
except AttributeError:
    pass  # fallback if not supported

nest_asyncio.apply()


# Step 2: configure OpenEvolve LLM
config = Config()
config.llm.models = [LLMModelConfig(
            api_base="https://openrouter.ai/api/v1",
            name="openrouter/free",
            api_key="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"
        )]
# Step 3: benchmark function
def benchmark_fib(path):
    spec = importlib.util.spec_from_file_location("candidate", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    start = time.time()
    module.fibonacci(20)
    end = time.time()

    return -(end - start)

# Step 4: async evaluator
def evaluator(path):
    try:
        return {"combined_score": benchmark_fib(path)}
    except:
        return {"combined_score": -1e9}


# Step 5: run OpenEvolve
if __name__ == "__main__":
    result = run_evolution(
        initial_program="""
def fibonacci(n):
    if n <= 1: return n
    return fibonacci(n-1) + fibonacci(n-2)
""",
        evaluator=evaluator,
        iterations=5,
        config=config
    )
    print("Best evolved Fibonacci code:\n", result.best_code.encode('utf-8', 'replace').decode('utf-8'))

      # This works

2026-03-19 01:26:29,010 - INFO - Logging to C:\Users\param\AppData\Local\Temp\openevolve_m8cr9xgf\logs\openevolve_20260319_012628.log
2026-03-19 01:26:29,016 - INFO - Set random seed to 42 for reproducibility
2026-03-19 01:26:29,221 - INFO - Initialized OpenAI LLM with model: openrouter/free
2026-03-19 01:26:29,222 - INFO - Initialized LLM ensemble with models: openrouter/free (weight: 1.00)
2026-03-19 01:26:29,223 - INFO - Initialized prompt sampler
2026-03-19 01:26:29,225 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-03-19 01:26:29,226 - INFO - Initialized program database with 0 programs
2026-03-19 01:26:29,228 - INFO - Successfully loaded evaluation function from C:\Users\param\AppData\Local\Temp\openevolve_m8cr9xgf\evaluator_550cc864.py
2026-03-19 01:26:29,228 - INFO - Initialized evaluator with C:\Users\param\AppData\Local\Temp\openevolve_m8cr9xgf\evaluator_550cc864.py
2026-03-19 01:26:29,228 - INFO - Initialized OpenEvolve with C:\Users\param\App

Best evolved Fibonacci code:
 # EVOLVE-BLOCK-START

def fibonacci(n):
    if n <= 1: return n
    return fibonacci(n-1) + fibonacci(n-2)

# EVOLVE-BLOCK-END


In [2]:
# evolve_bubblesort.py
import os
import sys
import io
import time
import importlib.util
import asyncio
from openevolve import evolve_function
from openevolve.config import Config, LLMModelConfig

# sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

# --- Configure OpenEvolve LLM ---
config = Config()
config.llm.models = [LLMModelConfig(name="gpt-4o-mini", api_key="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")]

# --- Benchmark / test cases for Bubble Sort ---
def benchmark_bubblesort(func, test_cases):
    """
    Run the candidate bubble_sort function on all test cases.
    Returns the number of successful test cases as the score.
    """
    score = 0
    for inp, expected in test_cases:
        try:
            result = func(inp.copy())
            if result == expected:
                score += 1
        except Exception:
            pass
    return score

# --- Async evaluator wrapper ---
async def evaluator(path):
    loop = asyncio.get_event_loop()
    score = await loop.run_in_executor(None, benchmark_bubblesort, path)
    return score

# --- Initial Bubble Sort function ---
def bubble_sort(arr):
    for i in range(len(arr)):
        for j in range(len(arr)-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

# --- Test cases ---
test_cases = [
    ([3, 1, 2], [1, 2, 3]),
    ([5, 2, 8], [2, 5, 8]),
    ([9, 7, 5, 3], [3, 5, 7, 9]),
    ([1], [1])
]


# --- Run OpenEvolve ---
if __name__ == "__main__":
    result = evolve_function(
        bubble_sort,
        test_cases=test_cases,
        iterations=10,      # increase for better evolution
        config=config,
        # evaluator=EVALUATOR
    )

    # Safe print
    print("Best evolved Bubble Sort code:\n", result.best_code)
    print("Score on test cases:", result.best_score)

    # This works

2026-03-19 01:31:11,228 - INFO - Logging to C:\Users\param\AppData\Local\Temp\openevolve_ed4olojq\logs\openevolve_20260319_013111.log
2026-03-19 01:31:11,228 - INFO - Logging to C:\Users\param\AppData\Local\Temp\openevolve_ed4olojq\logs\openevolve_20260319_013111.log
2026-03-19 01:31:11,229 - INFO - Set random seed to 42 for reproducibility
2026-03-19 01:31:11,229 - INFO - Set random seed to 42 for reproducibility
2026-03-19 01:31:11,417 - INFO - Initialized OpenAI LLM with model: gpt-4o-mini
2026-03-19 01:31:11,417 - INFO - Initialized OpenAI LLM with model: gpt-4o-mini
2026-03-19 01:31:11,420 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-03-19 01:31:11,420 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-03-19 01:31:11,421 - INFO - Initialized program database with 0 programs
2026-03-19 01:31:11,421 - INFO - Initialized program database with 0 programs
2026-03-19 01:31:11,424 - INFO - Successfully loaded evaluation functi

Best evolved Bubble Sort code:
 def bubble_sort(arr):
    # EVOLVE-BLOCK-START
    for i in range(len(arr)):
        for j in range(len(arr)-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

    # EVOLVE-BLOCK-END
Score on test cases: 0.0


In [ ]:
from openevolve import evolve_function
from openevolve.config import Config, LLMModelConfig

# --- Configure OpenEvolve LLM ---
config = Config()
config.llm.models = [
    LLMModelConfig(name="gpt-4o-mini", api_key="YOUR_API_KEY_HERE")
]

# --- Benchmark / test cases for Bubble Sort ---
def benchmark_bubblesort(func, test_cases):
    score = 0
    for inp, expected in test_cases:
        try:
            result = func(inp.copy())
            if result == expected:
                score += 1
        except Exception:
            pass
    return score

# --- Synchronous evaluator ---
def evaluator(func, test_cases):
    return benchmark_bubblesort(func, test_cases)

# --- Initial Bubble Sort function ---
def bubble_sort(arr):
    for i in range(len(arr)):
        for j in range(len(arr)-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

# --- Test cases ---
test_cases = [
    ([3, 1, 2], [1, 2, 3]),
    ([5, 2, 8], [2, 5, 8]),
    ([9, 7, 5, 3], [3, 5, 7, 9]),
    ([1], [1])
]

# --- Run evolution ---
if __name__ == "__main__":
    result = evolve_function(
        bubble_sort,
        test_cases=test_cases,
        iterations=10,      # increase for better evolution
        config=config,
        # evaluator=evaluator   # pass synchronous evaluator only
    )

    print("Best evolved Bubble Sort code:\n", result.best_code)
    print("Score on test cases:", result.best_score)

TypeError: run_evolution() got an unexpected keyword argument 'test_cases'